In [75]:
import pandas as pd
import numpy as np
import json
pd.set_option('display.max_columns', 500)

MARKET = "base_cbbtc_usdc_full"
# MARKET = "eth_cbbtc_usdc"
EVENTS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched"
HOURLY_MARKET_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_hourly_data"
POSITIONS_PATH = "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/users_positions/crypto_tokens_positions.csv"

df = pd.read_csv(f"{EVENTS_PATH}/{MARKET}.csv")
market_df = pd.read_csv(f"{HOURLY_MARKET_PATH}/{MARKET}.csv")
all_positions = pd.read_csv(POSITIONS_PATH)
positions = all_positions[all_positions["market"] == MARKET]
df.head(2)

,hash,type,timestamp,user_address,assets,assets_usd,liquidated_assets,liquidated_assets_usd,market,datetime,market_address,total_supply_before,total_borrow_before,total_supply_after,total_borrow_after,utilization_before,utilization_after,tx_actions,borrow_rate_before,supply_rate_before,borrow_rate_after,supply_rate_after,collateral_price,loan_asset_price,collateral_before,collateral_value_before,debt_before,supply_before,ltv_before,collateral_after,collateral_value_after,debt_after,supply_after,ltv_after,health_factor_before,health_factor_after,event_type,vault_flg,volatility_6h,drawdown_6h,trend_6h,volatility_24h,drawdown_24h,trend_24h,event_sequence_type,collateral_asset_symbol,loan_asset_symbol
0,0x4b95f45c3afdbdb97aca807ddc82d4d2b7cd2c09801b...,MarketSupply,1726217265,0xf8b9D83574574A345c32905a2A62D1d84650A0Ae,1000000,1.001,0,0.0,base_cbbtc_usdc,2024-09-13 08:47:45,0x9103c3b4e834476c9a62ea009ba2c884ee42e94e6e31...,0.0,0.0,1.000000,0.0,0.0,0.000000,1,0.002968,0.0,0.002968,0.000000,58132.0,1,0.0,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0,1.0,0.000000,0.0,0.000000,loan_position_supply,False,0.001659,-0.002464,-0.00422,0.003227,-0.002464,-0.000207,loan_position_supply,cbBTC,USDC
1,0xbc8e2030283aef18ec38474c5a6f62dbb20bc9d18688...,MarketBorrow,1726217705,0xf8b9D83574574A345c32905a2A62D1d84650A0Ae,1000000,1.001,0,0.0,base_cbbtc_usdc,2024-09-13 08:55:05,0x9103c3b4e834476c9a62ea009ba2c884ee42e94e6e31...,1.0,0.0,1.016478,1.0,0.0,0.983789,3,0.002968,0.0,0.041709,0.041041,58132.0,1,0.0,0.0,0.0,1.0,0.0,0.0001,5.8132,1.0,1.0,0.172022,0.0,4.999352,position_open,False,0.001659,-0.002464,-0.00422,0.003227,-0.002464,-0.000207,position_open,cbBTC,USDC


In [76]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [77]:
%autoreload 2
from utils import (
    select_users_by_period,
    create_hourly_user_dataset,
)


In [78]:

def get_users_with_leverage(df_actions, start_date, end_date, window_hours=24, threshold_date=None):
    filtered_actions = select_users_by_period(df_actions, start_date, end_date, threshold_date)
    
    open_events = filtered_actions[filtered_actions['event_sequence_type'] == 'position_open'][['user_address', 'timestamp']].copy()
    open_events = open_events.rename(columns={'timestamp': 'open_time'})
    
    leverage_counts = []
    for user in open_events['user_address'].unique():
        user_open = open_events[open_events['user_address'] == user]['open_time'].iloc[0]
        user_df = filtered_actions[filtered_actions['user_address'] == user]
        
        window_end = user_open + window_hours * 3600
        window_events = user_df[(user_df['timestamp'] >= user_open) & (user_df['timestamp'] <= window_end)]
        
        leverage_count = len(window_events[window_events['event_sequence_type'] == 'borrow_more_w_collateral'])
        leverage_counts.append({'user_address': user, 'leverage_factor': leverage_count})
    
    leverage_df = pd.DataFrame(leverage_counts)
    result = filtered_actions.merge(leverage_df, on='user_address', how='left')
    result["leverage_factor"] += 1

    if threshold_date is not None:
        result = result[result["datetime"].astype(str) < str(threshold_date)]
    
    return result


In [79]:
from tqdm.auto import tqdm
monthly_dfs = []
start = pd.Timestamp("2025-01-01")
end = pd.Timestamp("2025-12-01")
intervals = pd.date_range(start, end, freq=f'{1}MS')  # month starts

for i, interval_start in tqdm(enumerate(intervals)):
    interval_end = interval_start + pd.DateOffset(months=1)
    interval_thr = interval_start + pd.DateOffset(months=2)
    print(interval_start, interval_end)
    try:
        curr_df = get_users_with_leverage(df, interval_start, interval_end, 24, interval_thr)
    except Exception as e:
        print(e)
        continue
    monthly_dfs.append((f"{interval_start} - {interval_end}", curr_df))
    print(curr_df["datetime"].min(), curr_df["datetime"].max())


0it [00:00, ?it/s]

2025-01-01 00:00:00 2025-02-01 00:00:00
2025-01-01 00:58:01 2025-02-28 23:06:39
2025-02-01 00:00:00 2025-03-01 00:00:00
2025-02-01 00:09:01 2025-03-31 23:48:29
2025-03-01 00:00:00 2025-04-01 00:00:00
2025-03-01 00:47:07 2025-04-30 23:15:43
2025-04-01 00:00:00 2025-05-01 00:00:00
2025-04-01 00:16:41 2025-05-31 23:59:45
2025-05-01 00:00:00 2025-06-01 00:00:00
2025-05-01 00:23:41 2025-06-30 23:44:05
2025-06-01 00:00:00 2025-07-01 00:00:00
2025-06-01 00:15:13 2025-07-31 23:59:29
2025-07-01 00:00:00 2025-08-01 00:00:00
2025-07-01 00:06:13 2025-08-31 23:52:43
2025-08-01 00:00:00 2025-09-01 00:00:00
2025-08-01 00:04:35 2025-09-30 23:50:53
2025-09-01 00:00:00 2025-10-01 00:00:00
2025-09-01 01:01:59 2025-10-31 23:18:23
2025-10-01 00:00:00 2025-11-01 00:00:00
2025-10-01 00:00:11 2025-11-26 17:20:07
2025-11-01 00:00:00 2025-12-01 00:00:00
2025-11-01 00:00:59 2025-11-26 17:12:33
2025-12-01 00:00:00 2026-01-01 00:00:00
'user_address'
11


  0%|          | 0/11 [00:00<?, ?it/s]

KeyError: 'large'

In [81]:

cohort_clusters = {
    "all": [],
    "risky": [],
    "large": [],
    
}
print(len(monthly_dfs))

for inx in tqdm(range(len(monthly_dfs))):
    i,curr_df  = monthly_dfs[inx]
    risky_users = curr_df[curr_df["ltv_after"] > 0.8]["user_address"].unique() 
    large_users = curr_df[curr_df["debt_after"] > 300_000]["user_address"].unique() 
    small_ltv = curr_df.groupby("user_address")["ltv_after"].max()
    small_ltv = small_ltv[small_ltv<0.6].index
    small_size = curr_df.groupby("user_address")["debt_after"].max()
    small_size = small_size[small_size<10_000].index

    cohort_clusters["all"].append(curr_df)
    cohort_clusters["risky"].append(
        curr_df[
            curr_df["user_address"].isin( curr_df[curr_df["debt_after"] > 10_000]["user_address"].unique() ) &
            curr_df["user_address"].isin( risky_users )
        ]
    )
    cohort_clusters["large"].append(
        curr_df[
            curr_df["user_address"].isin( large_users )
        ]
    )
    print(i, curr_df.shape, cohort_clusters["risky"][-1].shape, cohort_clusters["large"][-1].shape)


11


  0%|          | 0/11 [00:00<?, ?it/s]

2025-01-01 00:00:00 - 2025-02-01 00:00:00 (5375, 48) (1331, 48) (1202, 48)
2025-02-01 00:00:00 - 2025-03-01 00:00:00 (8124, 48) (613, 48) (309, 48)
2025-03-01 00:00:00 - 2025-04-01 00:00:00 (14527, 48) (222, 48) (388, 48)
2025-04-01 00:00:00 - 2025-05-01 00:00:00 (19273, 48) (205, 48) (1050, 48)
2025-05-01 00:00:00 - 2025-06-01 00:00:00 (25011, 48) (367, 48) (1225, 48)
2025-06-01 00:00:00 - 2025-07-01 00:00:00 (19336, 48) (264, 48) (1413, 48)
2025-07-01 00:00:00 - 2025-08-01 00:00:00 (21928, 48) (1207, 48) (1815, 48)
2025-08-01 00:00:00 - 2025-09-01 00:00:00 (18459, 48) (463, 48) (727, 48)
2025-09-01 00:00:00 - 2025-10-01 00:00:00 (21835, 48) (1304, 48) (1047, 48)
2025-10-01 00:00:00 - 2025-11-01 00:00:00 (28563, 48) (2116, 48) (2182, 48)
2025-11-01 00:00:00 - 2025-12-01 00:00:00 (18509, 48) (230, 48) (446, 48)


In [82]:
cohort_clusters_hourly = {}
for cluster in list(cohort_clusters.keys())[1:]:
    cohort_clusters_hourly[cluster] = []
    for curr_df in tqdm(cohort_clusters[cluster]):
        thr = str(curr_df["datetime"].max())[:10]
        # print(str(thr)[:10], curr_df["datetime"].max())
        hourly = create_hourly_user_dataset(
            curr_df,
            market_df,
            n_hours=1,
            threshold_date=str(thr[:10]),
        )
        cohort_clusters_hourly[cluster].append(hourly)


  0%|          | 0/11 [00:00<?, ?it/s]

  0%|          | 0/11 [00:00<?, ?it/s]

In [52]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np


def remove_similar_hours(hourly_df, diff_thresholds, min_events_cnt=5):
    df = hourly_df.sort_values(['user_address', 'timestamp']).copy()
    processed_thresholds = {}
    threshold_types = {}
    for col, thresh in diff_thresholds.items():
        if isinstance(thresh, str) and thresh.endswith('%'):
            processed_thresholds[col] = float(thresh.strip('%')) / 100
            threshold_types[col] = 'relative'
        else:
            processed_thresholds[col] = float(thresh)
            threshold_types[col] = 'absolute'

    result_dfs = []
    for user, user_df in df.groupby('user_address'):
        user_df = user_df.sort_values('timestamp')
        action_mask = user_df['action'] != 'none'
        keep_indices = set(user_df[action_mask].index)
        last_kept_idx = None

        for idx, curr_row in user_df.iterrows():
            if idx in keep_indices or last_kept_idx is None:
                keep_indices.add(idx)
                last_kept_idx = idx
                continue

            prev_row = user_df.loc[last_kept_idx]
            similar = True
            for col, thresh in processed_thresholds.items():
                if col not in curr_row or col not in prev_row:
                    continue
                curr_val = curr_row[col]
                prev_val = prev_row[col]
                if threshold_types[col] == 'relative':
                    if prev_val == 0:
                        diff = abs(curr_val - prev_val)
                    else:
                        diff = abs((curr_val - prev_val) / prev_val)
                else:
                    diff = abs(curr_val - prev_val)

                if diff > thresh:
                    similar = False
                    break
            if not similar:
                keep_indices.add(idx)
                last_kept_idx = idx

        if len(keep_indices) < min_events_cnt:
            keep_indices = set(user_df[action_mask].index)

        if keep_indices:
            result_dfs.append(user_df.loc[list(keep_indices)])

    return pd.concat(result_dfs).sort_values(['user_address', 'timestamp'])


def add_time_difference_features(hourly_df, fields=None, windows=None):
    if fields is None:
        fields = ['ltv', 'collateral_price', 'borrow_rate', 'market_utilization']
    if windows is None:
        windows = [6, 24]

    df = hourly_df.sort_values(['user_address', 'timestamp']).copy()

    open_info = df.groupby('user_address')['timestamp'].min().reset_index()
    open_info.columns = ['user_address', 'open_timestamp']
    df = df.merge(open_info, on='user_address', how='left')

    for field in fields:
        open_vals = df[df['timestamp'] == df['open_timestamp']].set_index('user_address')[field]
        df[f'{field}_change_open'] = df['user_address'].map(open_vals)
        df[f'{field}_change_open'] = df[field] - df[f'{field}_change_open']

    for w in windows:
        for field in fields:
            shifted = df.groupby('user_address')[field].shift(w)
            open_vals = df.groupby('user_address')[field].transform('first')
            filled = shifted.fillna(open_vals)
            df[f'{field}_change_{w}h'] = df[field] - filled

    df.drop(columns=['open_timestamp'], inplace=True)
    return df


def create_model_dataset(
    raw_hourly_df,
    target_horizon_hours=4,
    leverage_factor_min=1,
    liq_threshold=0.86,
    diff_thresholds=None,
    min_events_cnt=5,
    diff_fields=None,
    diff_windows=None
):
    """
    Full pipeline: remove similar hours, add time‑difference features, engineer model features.
    """
    if diff_fields is None:
        diff_fields = ['ltv', 'collateral_price', 'borrow_rate', 'market_utilization']
    if diff_windows is None:
        diff_windows = [6, 24]

    df = raw_hourly_df.copy()

    if diff_thresholds is not None:
        df = remove_similar_hours(df, diff_thresholds, min_events_cnt=min_events_cnt)

    df = add_time_difference_features(df, fields=diff_fields, windows=diff_windows)

    if leverage_factor_min > 0 and 'leverage_factor' in df.columns:
        df = df[df['leverage_factor'] >= leverage_factor_min]

    df['log_debt'] = np.log1p(df['debt'])
    df['has_action'] = (df['action'] != 'none').astype(int)

    open_times = df.groupby('user_address')['timestamp'].min().rename('open_timestamp')
    df = df.join(open_times, on='user_address')
    df['hours_since_open'] = (df['timestamp'] - df['open_timestamp']) / 3600.0
    df.drop(columns=['open_timestamp'], inplace=True)

    def compute_hours_since_last_action(group):
        group = group.sort_values('timestamp')
        last_action_ts = None
        res = []
        for ts, action in zip(group['timestamp'], group['has_action']):
            if action == 1:
                res.append(0.0)
                last_action_ts = ts
            else:
                if last_action_ts is None:
                    res.append(1000.0)
                else:
                    res.append((ts - last_action_ts) / 3600.0)
        return pd.Series(res, index=group.index)

    df['hours_since_last_action'] = df.groupby('user_address', group_keys=False).apply(
        compute_hours_since_last_action
    )

    def action_count_last_n_hours(group, window_hours=6):
        group = group.sort_values('timestamp')
        timestamps = group['timestamp'].values
        actions = group['has_action'].values
        counts = []
        for i, ts in enumerate(timestamps):
            start = ts - window_hours * 3600
            left = np.searchsorted(timestamps, start, side='left')
            counts.append(np.sum(actions[left:i]))
        return pd.Series(counts, index=group.index)

    df['action_count_last_6h'] = df.groupby('user_address', group_keys=False).apply(
        lambda g: action_count_last_n_hours(g, 6)
    )

    df['ltv_change_1h'] = df.groupby('user_address')['ltv'].diff(1)
    df['distance_to_liq'] = np.maximum(liq_threshold - df['ltv'], 0)

    if 'borrow_rate_rolling' in df.columns:
        df['borrow_rate_trend'] = df['borrow_rate'] - df['borrow_rate_rolling']
    else:
        df['borrow_rate_trend'] = df.groupby('user_address')['borrow_rate'].transform(
            lambda x: x - x.rolling(6, min_periods=1).mean()
        )

    df['ltv_times_rate'] = df['ltv'] * df['borrow_rate']
    df['debt_change_1h'] = df.groupby('user_address')['debt'].diff(1)
    df['volatility_ltv'] = df['volatility_6h'] * df['ltv']

    df['hour_of_day'] = pd.to_datetime(df['timestamp'], unit='s').dt.hour
    df['day_of_week'] = pd.to_datetime(df['timestamp'], unit='s').dt.dayofweek

    def has_action_in_future(group):
        res = []
        timestamps = group['timestamp'].values
        actions = group['has_action'].values
        for i in range(len(group)):
            current_ts = timestamps[i]
            horizon_ts = current_ts + target_horizon_hours * 3600
            future_mask = (timestamps > current_ts) & (timestamps <= horizon_ts)
            res.append(1 if np.any(actions[future_mask]) else 0)
        return pd.Series(res, index=group.index)

    df['action_next'] = df.groupby('user_address', group_keys=False).apply(has_action_in_future)

    diff_feature_cols = []
    for field in diff_fields:
        diff_feature_cols.append(f'{field}_change_open')
        for w in diff_windows:
            diff_feature_cols.append(f'{field}_change_{w}h')

    base_features = [
        'hours_since_open',
        'hours_since_last_action',
        'action_count_last_6h',
        'ltv_change_1h',
        'distance_to_liq',
        'borrow_rate_trend',
        'ltv_times_rate',
        'debt_change_1h',
        'volatility_ltv',
        'hour_of_day',
        'day_of_week',
        'log_debt'
    ]

    all_features = diff_feature_cols + base_features
    available_features = [f for f in all_features if f in df.columns]

    return df[['user_address', 'timestamp'] + available_features + ['action_next']].fillna(0)

In [59]:
diff_thresholds = {
    'collateral_price': '5%',
    'market_utilization': 0.02,
    'borrow_rate': 0.02,
}

model_df = create_model_dataset(
    raw_hourly_df=cohort_clusters_hourly['risky'][0],
    target_horizon_hours=4,
    leverage_factor_min=1,
    liq_threshold=0.86,
    diff_thresholds=diff_thresholds,
    min_events_cnt=5,
    diff_fields=['ltv', 'collateral_price', 'borrow_rate', 'market_utilization'],
    diff_windows=[6, 24]
)
model_df.head(4)

/Users/yegortrussov/Library/Python/3.9/lib/python/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/var/folders/hj/pbs977kd43s6n1l9z3mxrj200000gn/T/ipykernel_40486/738352471.py:142: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['hours_since_last_action'] = df.groupby('user_address', group_keys=False).apply(
/var/folders/hj/pbs977kd43s6n1l9z3mxrj200000gn/T/ipykernel_40486/738352471.py:157: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `

,user_address,timestamp,ltv_change_open,ltv_change_6h,ltv_change_24h,collateral_price_change_open,collateral_price_change_6h,collateral_price_change_24h,borrow_rate_change_open,borrow_rate_change_6h,borrow_rate_change_24h,market_utilization_change_open,market_utilization_change_6h,market_utilization_change_24h,hours_since_open,hours_since_last_action,action_count_last_6h,ltv_change_1h,distance_to_liq,borrow_rate_trend,ltv_times_rate,debt_change_1h,volatility_ltv,hour_of_day,day_of_week,log_debt,action_next
0,0x0EDD42c2888d9Ecd7fBc11C657354953f33eEf67,1735949475,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0,0.000000,0.068304,-0.010230,0.057602,0.000000,0.002844,0,5,10.998505,1
1,0x0EDD42c2888d9Ecd7fBc11C657354953f33eEf67,1735953075,-0.002539,-0.002539,-0.002539,-67.0,-67.0,-67.0,-0.000121,-0.000121,-0.000121,-0.002307,-0.002307,-0.002307,1.0,0.0,1,-0.002539,0.070842,-0.006714,0.057322,-232.415310,0.002685,1,5,10.994610,1
2,0x0EDD42c2888d9Ecd7fBc11C657354953f33eEf67,1735963875,-0.011515,-0.011515,-0.011515,180.0,180.0,180.0,0.003949,0.003949,0.003949,0.009973,0.009973,0.009973,4.0,0.0,2,-0.008976,0.079818,-0.002451,0.059846,-528.961855,0.001497,4,5,10.985688,0
3,0x0EDD42c2888d9Ecd7fBc11C657354953f33eEf67,1735989075,-0.011724,-0.011724,-0.011724,-335.0,-335.0,-335.0,-0.000109,-0.000109,-0.000109,-0.004245,-0.004245,-0.004245,11.0,0.0,0,-0.000209,0.080028,-0.006032,0.056664,-27115.000000,0.000884,11,5,10.370622,0


In [88]:
import pandas as pd
import numpy as np
import warnings
import statsmodels.api as sm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, classification_report
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
def print_model_summaries(summaries, names=None, top_n=10, verbose=True):
    if names is None:
        names = [f"Model {i+1}" for i in range(len(summaries))]

    all_feature_dfs = []

    for name, summary in zip(names, summaries):
        metrics = summary['metrics']
        coef = summary['feature_importance'].copy()
        coef['model_name'] = name
        all_feature_dfs.append(coef)

        if verbose:
            print(f"### {name}")
            print(f"ROC-AUC: {metrics['roc_auc']:.4f} | PR-AUC: {metrics['pr_auc']:.4f} | "
                  f"Recall: {metrics['recall']:.4f} | F1: {metrics['f1_score']:.4f}")
            print(f"| {'Feature':<30} | {'Coef':>7} | {'p-val':>8} | Sig |")
            print("|-" + "-"*30 + "-|-" + "-"*7 + "-|-" + "-"*8 + "-|---|")

            for _, row in coef.head(top_n).iterrows():
                sig_str = "*" if row['significant'] else ""
                print(f"| {row['feature']:<30} | {row['coefficient']:7.4f} | {row['p_value']:8.4f} | {sig_str:>3} |")
            print()

    if not all_feature_dfs:
        return pd.DataFrame()

    combined = pd.concat(all_feature_dfs, ignore_index=True)

    global_summary = combined.groupby('feature').agg(
        count=('coefficient', 'count'),
        mean_coef=('coefficient', 'mean'),
        mean_abs_coef=('coefficient', lambda x: np.mean(np.abs(x))),
        mean_pval=('p_value', 'mean'),
        prop_significant=('significant', 'mean'),
        mode_sign=('sign', lambda x: x.mode().iloc[0] if not x.mode().empty else '?')
    ).reset_index()

    global_summary = global_summary.sort_values('prop_significant', ascending=False)

    if verbose:
        print("### GLOBAL FEATURE SIGNIFICANCE SUMMARY")
        print(f"| {'Feature':<30} | {'Count':>5} | {'Mean Coef':>9} | "
              f"{'Mean p-val':>9} | {'Prop Sig':>8} | {'Sign':>4} |")
        print("|-" + "-"*30 + "-|-" + "-"*5 + "-|-" + "-"*9 + "-|-" + "-"*9 + "-|-" + "-"*8 + "-|-" + "-"*4 + "-|")

        for _, row in global_summary.iterrows():
            print(f"| {row['feature']:<30} | {row['count']:5d} | {row['mean_coef']:9.4f} | "
                  f"{row['mean_pval']:9.4f} | {row['prop_significant']:8.2f} | {row['mode_sign']:>4} |")

    return global_summary
def train_and_summarize_model(model_df,
                              target_col='action_next',
                              remove_features=None,
                              split_type='time',
                              test_size=0.2,
                              random_state=42,
                              scale_features=True,
                              verbose=True,
                              add_derived_features=True):
    if remove_features is None:
        remove_features = []
    df = model_df.copy()

    categorical_cols = []  # can be passed as argument if needed
    feature_cols = [col for col in df.columns if col not in ['user_address', 'timestamp', target_col]]
    for col in categorical_cols:
        if col in feature_cols:
            feature_cols.remove(col)
    feature_cols = [col for col in feature_cols if col not in remove_features]

    if add_derived_features:
        if 'ltv' in df.columns and 'ltv' not in remove_features:
            df['ltv_squared'] = df['ltv'] ** 2
            if 'ltv_squared' not in remove_features:
                feature_cols.append('ltv_squared')
        if ('ltv' in df.columns and 'ltv' not in remove_features and
            'hours_since_open' in df.columns and 'hours_since_open' not in remove_features):
            df['ltv_time'] = df['ltv'] * df['hours_since_open']
            if 'ltv_time' not in remove_features:
                feature_cols.append('ltv_time')
        if ('ltv' in df.columns and 'ltv' not in remove_features and
            'borrow_rate' in df.columns and 'borrow_rate' not in remove_features):
            df['ltv_times_rate'] = df['ltv'] * df['borrow_rate']
            if 'ltv_times_rate' not in remove_features:
                feature_cols.append('ltv_times_rate')

    if categorical_cols:
        dummies = pd.get_dummies(df[categorical_cols], drop_first=True)
        dummy_names = [dn for dn in dummies.columns if dn not in remove_features]
        dummies = dummies[dummy_names]
    else:
        dummies = pd.DataFrame(index=df.index)
        dummy_names = []

    X = pd.concat([df[feature_cols], dummies], axis=1)
    y = df[target_col].copy()
    final_feature_names = feature_cols + dummy_names

    if split_type == 'time':
        split_idx = int(len(X) * (1 - test_size))
        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
    else:
        from sklearn.model_selection import train_test_split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state, stratify=y
        )

    X_train = X_train.replace([np.inf, -np.inf], np.nan)
    X_test = X_test.replace([np.inf, -np.inf], np.nan)
    for col in X_train.columns:
        median_val = X_train[col].median()
        X_train[col].fillna(median_val, inplace=True)
        X_test[col].fillna(median_val, inplace=True)

    scaler = StandardScaler() if scale_features else None
    if scale_features:
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
    else:
        X_train_scaled = X_train.values
        X_test_scaled = X_test.values

    X_train_sm = sm.add_constant(X_train_scaled)
    X_test_sm = sm.add_constant(X_test_scaled)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model_sm = sm.Logit(y_train, X_train_sm)
        result = model_sm.fit(disp=0, method='bfgs', maxiter=1000)

    full_feature_names = ['const'] + final_feature_names
    result.params.index = full_feature_names
    result.bse.index = full_feature_names

    y_prob = result.predict(X_test_sm)
    y_pred = (y_prob >= 0.5).astype(int)

    metrics = {
        'roc_auc': roc_auc_score(y_test, y_prob),
        'pr_auc': average_precision_score(y_test, y_prob),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred, zero_division=0)
    }

    coef_df = pd.DataFrame({
        'feature': full_feature_names,
        'coefficient': result.params.values,
        'p_value': result.pvalues.values,
        'std_err': result.bse.values
    })
    coef_df = coef_df[coef_df['feature'] != 'const']
    coef_df['odds_ratio'] = np.exp(coef_df['coefficient'])
    coef_df['significant'] = coef_df['p_value'] < 0.05
    coef_df['sign'] = np.where(coef_df['coefficient'] > 0, '+', '-')
    coef_df = coef_df.sort_values('p_value', ascending=True)

    if verbose:
        print("\n" + "=" * 60)
        print("MODEL PERFORMANCE")
        print("=" * 60)
        print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
        print(f"Action rate in test: {y_test.mean():.3f}")
        for m, v in metrics.items():
            print(f"{m:<15}: {v:.4f}")
        print("\n" + "=" * 60)
        print("FEATURE SIGNIFICANCE (sorted by p-value)")
        print("=" * 60)
        print(coef_df[['feature', 'coefficient', 'p_value', 'significant', 'sign']].to_string(index=False))

        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, label=f'ROC (AUC = {metrics["roc_auc"]:.3f})')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curve')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    summary = {
        'metrics': metrics,
        'feature_importance': coef_df
    }
    return result, summary

def train_multiple_models(model_dfs, names=None,
                          target_col='action_next', test_size=0.2,
                          random_state=42, scale_features=False,
                          verbose=False):
    """
    Trains a logistic regression model for each DataFrame in the list,
    collects feature importance and significance.

    Parameters
    ----------
    model_dfs : list of pd.DataFrame
        List of prepared datasets.
    names : list of str, optional
        Names for each model, used as identifiers. If None, indices are used.
    target_col : str
        Name of the target column.
    test_size : float
        Fraction for test split.
    random_state : int
        Random seed.
    scale_features : bool
        Whether to standardize features.
    verbose : bool
        If True, prints progress.

    Returns
    -------
    all_features_df : pd.DataFrame
        Combined DataFrame with columns: model_name, feature, coefficient,
        odds_ratio, p_value, significant.
    models : list
        List of trained model results.
    summaries : list
        List of summary dicts.
    """
    if names is None:
        names = [f"model_{i}" for i in range(len(model_dfs))]

    all_features = []
    models = []
    summaries = []

    for name, df in zip(names, model_dfs):
        if verbose:
            print(f"Training model: {name}")
        _, summary = train_and_summarize_model(
            df,
            target_col=target_col,
            test_size=test_size,
            random_state=random_state,
            scale_features=scale_features,
            verbose=False
        )
        feat_df = summary['feature_importance'].copy()
        feat_df['model_name'] = name
        all_features.append(feat_df)
        models.append(summary)
        summaries.append(summary)

    all_features_df = pd.concat(all_features, ignore_index=True)
    all_features_df = all_features_df[['model_name', 'feature', 'coefficient',
                                       'odds_ratio', 'p_value', 'significant']]

    return all_features_df, models, summaries

diff_thresholds = {
    'collateral_price': '5%',
    'market_utilization': 0.02,
    'borrow_rate': 0.02,
}

models_dfs = []
for x in cohort_clusters_hourly['large']:
    df = create_model_dataset(
        raw_hourly_df=x,
        target_horizon_hours=4,
        leverage_factor_min=1,
        liq_threshold=0.86,
        diff_thresholds=diff_thresholds,
        min_events_cnt=5,
        diff_fields=['ltv', 'collateral_price', 'borrow_rate', 'market_utilization'],
        diff_windows=[6, 24, 72]
    )
    df = df.drop(columns=[
        "collateral_price_change_open",
        "borrow_rate_change_6h",
        "borrow_rate_change_72h",
        # "borrow_rate_change_open",
        "borrow_rate_trend",
    ])
    models_dfs.append(df)

interval_month_len = 4
models_dfs_prep = []
for i in range(len(models_dfs) - interval_month_len + 1):
    curr_df = pd.concat(models_dfs[i:i + interval_month_len], ignore_index=True)
    models_dfs_prep.append(curr_df)

names = [f"window_{i}" for i in range(len(models_dfs_prep))]

all_features_df, models, summaries = train_multiple_models(
    models_dfs_prep,
    names=names,
    target_col='action_next',
    test_size=0.2,
    scale_features=True,
    verbose=False,
)

print_model_summaries(summaries, names=names)


### window_0
ROC-AUC: 0.7949 | PR-AUC: 0.4708 | Recall: 0.0160 | F1: 0.0309
| Feature                        |    Coef |    p-val | Sig |
|--------------------------------|---------|----------|---|
| hours_since_last_action        | -1.9052 |   0.0000 |   * |
| log_debt                       | -0.2954 |   0.0000 |   * |
| action_count_last_6h           |  0.1977 |   0.0000 |   * |
| distance_to_liq                | -0.3480 |   0.0001 |   * |
| hour_of_day                    |  0.1820 |   0.0011 |   * |
| collateral_price_change_6h     | -0.1673 |   0.0228 |   * |
| collateral_price_change_72h    |  0.2134 |   0.0322 |   * |
| borrow_rate_change_24h         | -0.1092 |   0.2796 |     |
| volatility_ltv                 |  0.0548 |   0.3373 |     |
| market_utilization_change_72h  |  0.1036 |   0.3655 |     |

### window_1
ROC-AUC: 0.8616 | PR-AUC: 0.5016 | Recall: 0.3208 | F1: 0.4000
| Feature                        |    Coef |    p-val | Sig |
|--------------------------------|---------

,feature,count,mean_coef,mean_abs_coef,mean_pval,prop_significant,mode_sign
0,action_count_last_6h,8,0.257902,0.257902,1.689720e-04,1.000,+
10,hours_since_last_action,8,-2.126389,2.126389,7.175998e-11,1.000,-
8,distance_to_liq,8,-0.398797,0.440191,1.156707e-01,0.750,-
11,hours_since_open,8,-0.198985,0.201517,1.349197e-01,0.750,-
12,log_debt,8,-0.201003,0.201003,6.997471e-02,0.750,-
5,collateral_price_change_72h,8,0.347847,0.371187,2.229103e-01,0.625,+
13,ltv_change_1h,8,-0.183311,0.195268,1.393693e-01,0.625,-
1,borrow_rate_change_24h,8,-0.341941,0.418174,2.052392e-01,0.500,-
17,ltv_change_open,8,0.964822,0.974267,3.692330e-01,0.500,+
4,collateral_price_change_6h,8,-0.189087,0.189087,9.582351e-02,0.500,-


In [92]:
models_dfs_prep[0]['distance_to_liq'].describe()

count    4492.000000
mean        0.259113
std         0.177410
min         0.000000
25%         0.155034
50%         0.249016
75%         0.346057
max         3.708377
Name: distance_to_liq, dtype: float64

In [73]:
summaries[0]['feature_importance']['feature'].unique()

array(['hours_since_last_action', 'collateral_price_change_72h',
       'action_count_last_6h', 'hour_of_day',
       'collateral_price_change_6h', 'borrow_rate_change_open',
       'market_utilization_change_open', 'borrow_rate_change_72h',
       'market_utilization_change_72h', 'borrow_rate_trend',
       'distance_to_liq', 'day_of_week', 'volatility_ltv',
       'ltv_change_24h', 'collateral_price_change_24h',
       'borrow_rate_change_24h', 'hours_since_open', 'ltv_change_1h',
       'ltv_change_72h', 'log_debt', 'market_utilization_change_6h',
       'ltv_change_open', 'market_utilization_change_24h',
       'borrow_rate_change_6h', 'debt_change_1h', 'ltv_change_6h',
       'ltv_times_rate'], dtype=object)

In [83]:
def evaluate_target_horizons(
    raw_datasets_list,
    horizons,
    interval_month_len=3,
    diff_thresholds=None,
    min_events_cnt=5,
    diff_fields=['ltv', 'collateral_price', 'borrow_rate', 'market_utilization'],
    diff_windows=[6, 24],
    leverage_factor_min=1,
    liq_threshold=0.86,
    split_type='time',
    test_size=0.2,
    scale_features=True,
    add_derived_features=True,
    remove_features=None,
    random_state=42,
    verbose=True,
    top_n=15
):
    """
    Evaluates different target_horizon_hours values for elasticity modeling.

    For each horizon, trains models on sliding windows of concatenated datasets,
    aggregates performance metrics, and for the best horizon returns a detailed
    feature significance summary with odds ratios.

    Returns
    -------
    best_horizon : float
        The horizon value that maximized mean ROC-AUC.
    best_feature_df : pd.DataFrame
        Feature significance aggregated across windows for the best horizon.
    all_metrics_df : pd.DataFrame
        Mean and std of metrics for every horizon.
    """
    horizon_metrics = {}
    horizon_summaries = {}

    for horizon in horizons:
        if verbose:
            print(f"\n=== Target horizon = {horizon}h ===")

        models_dfs = []
        for raw_df in raw_datasets_list:
            df = create_model_dataset(
                raw_hourly_df=raw_df,
                target_horizon_hours=horizon,
                leverage_factor_min=leverage_factor_min,
                liq_threshold=liq_threshold,
                diff_thresholds=diff_thresholds,
                min_events_cnt=min_events_cnt,
                diff_fields=diff_fields,
                diff_windows=diff_windows
            )
            models_dfs.append(df)

        windowed_dfs = []
        for i in range(len(models_dfs) - interval_month_len + 1):
            windowed_dfs.append(
                pd.concat(models_dfs[i:i+interval_month_len], ignore_index=True)
            )

        names = [f"w{i}" for i in range(len(windowed_dfs))]

        _, _, summaries = train_multiple_models(
            windowed_dfs,
            names=names,
            target_col='action_next',
            test_size=test_size,
            random_state=random_state,
            scale_features=scale_features,
            verbose=False,
        )

        horizon_summaries[horizon] = summaries

        metrics_list = pd.DataFrame([s['metrics'] for s in summaries])
        agg_metrics = metrics_list.agg(['mean', 'std']).T
        agg_metrics.columns = ['mean', 'std']

        if verbose:
            print("\n--- Mean metrics across windows ---")
            print(agg_metrics.to_string(float_format=lambda x: f"{x:.4f}"))

        horizon_metrics[horizon] = agg_metrics

    if not horizon_metrics:
        return None, None, pd.DataFrame()

    all_metrics_df = pd.concat(
        [m.assign(horizon=h) for h, m in horizon_metrics.items()]
    ).reset_index().rename(columns={'index': 'metric'})
    all_metrics_df = all_metrics_df.pivot(index='metric', columns='horizon', values=['mean','std'])

    best_horizon = max(horizon_metrics.keys(), key=lambda h: horizon_metrics[h].loc['roc_auc','mean'])

    if verbose:
        print(f"\n=== Best horizon = {best_horizon}h (by mean ROC-AUC) ===")

    best_summaries = horizon_summaries[best_horizon]
    all_features = pd.concat(
        [s['feature_importance'] for s in best_summaries],
        ignore_index=True
    )

    feature_agg = all_features.groupby('feature').agg(
        mean_coef=('coefficient', 'mean'),
        mean_odds_ratio=('odds_ratio', 'mean'),
        mean_pval=('p_value', 'mean'),
        prop_significant=('significant', 'mean'),
        mode_sign=('sign', lambda x: x.mode().iloc[0] if not x.mode().empty else '?'),
        n_models=('coefficient', 'count')
    ).reset_index()

    feature_agg = feature_agg.sort_values('mean_pval', ascending=True)

    if verbose:
        print("\n=== Aggregated Feature Significance ===")
        print(f"| {'Feature':<30} | {'Coef':>7} | {'OddsRatio':>9} | {'p-val':>7} | {'Sig%':>5} | Sign |")
        print("|-" + "-"*30 + "-|-" + "-"*7 + "-|-" + "-"*9 + "-|-" + "-"*7 + "-|-" + "-"*5 + "-|-" + "-"*4 + "-|")
        for _, row in feature_agg.head(top_n).iterrows():
            print(f"| {row['feature']:<30} | {row['mean_coef']:7.4f} | {row['mean_odds_ratio']:9.4f} | "
                  f"{row['mean_pval']:7.4f} | {row['prop_significant']:5.2f} | {row['mode_sign']:>4} |")

    return best_horizon, feature_agg, all_metrics_df

diff_thresholds = {
    'collateral_price': '5%',
    'market_utilization': 0.02,
    'borrow_rate': 0.02,
}
interval_month_len = 4
horizons = [1, 4, 8, 12, 24]

best_horizon, best_features, all_metrics = evaluate_target_horizons(
    raw_datasets_list=cohort_clusters_hourly['large'],
    horizons=horizons,
    interval_month_len=interval_month_len,
    diff_thresholds=diff_thresholds,
    min_events_cnt=5,
    leverage_factor_min=1,
    liq_threshold=0.86,
    split_type='time',
    test_size=0.2,
    scale_features=True,
    add_derived_features=True,
    random_state=42,
    verbose=True,
    top_n=15
)

print(f"\nBest horizon: {best_horizon}h")
print(f"Top feature: {best_features['feature'].iloc[0]}, OR={best_features['mean_odds_ratio'].iloc[0]:.4f}")



=== Target horizon = 1h ===

--- Mean metrics across windows ---
            mean    std
roc_auc   0.7679 0.0538
pr_auc    0.2467 0.0800
accuracy  0.8911 0.0369
precision 0.2465 0.1921
recall    0.0722 0.0950
f1_score  0.0984 0.1090

=== Target horizon = 4h ===

--- Mean metrics across windows ---
            mean    std
roc_auc   0.7832 0.0492
pr_auc    0.4359 0.1125
accuracy  0.8264 0.0438
precision 0.5284 0.1066
recall    0.2213 0.1646
f1_score  0.2893 0.1744

=== Target horizon = 8h ===

--- Mean metrics across windows ---
            mean    std
roc_auc   0.7821 0.0540
pr_auc    0.5249 0.1326
accuracy  0.7970 0.0400
precision 0.6036 0.1343
recall    0.2940 0.1535
f1_score  0.3856 0.1654

=== Target horizon = 12h ===

--- Mean metrics across windows ---
            mean    std
roc_auc   0.7864 0.0510
pr_auc    0.5822 0.1256
accuracy  0.7832 0.0445
precision 0.6638 0.1158
recall    0.3537 0.1528
f1_score  0.4533 0.1556

=== Target horizon = 24h ===

--- Mean metrics across windows 

In [85]:
best_horizon

24